In [1]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found. Check your project-root .env file.")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=api_key,
    temperature=0
)

print("LangChain Groq model initialized successfully.")

LangChain Groq model initialized successfully.


In [2]:
class InterviewAnswer(BaseModel):
    question: str = Field(description="Interview question asked")
    answer: str = Field(description="Candidate answer, or a note that it was not provided")


class ExtractedAnswers(BaseModel):
    answers: list[InterviewAnswer] = Field(
        description="Interview questions and their corresponding candidate answers"
    )


class SkillEvaluation(BaseModel):
    skill: str = Field(description="Skill being assessed")
    demonstrated: bool = Field(
        description="True only when the transcript contains clear evidence of the skill"
    )
    evidence_or_gap: str = Field(
        description="Specific transcript evidence, or explanation of missing evidence"
    )


class SkillEvaluations(BaseModel):
    evaluations: list[SkillEvaluation] = Field(
        description="Evaluation of each requested skill"
    )


class ScorecardRow(BaseModel):
    skill: str
    demonstrated: bool
    evidence_or_gap: str


class CandidateScorecard(BaseModel):
    scorecard_rows: list[ScorecardRow]
    overall_recommendation: str = Field(
        description="Evidence-based recommendation, noting uncertainty and missing evidence"
    )


answers_parser = PydanticOutputParser(pydantic_object=ExtractedAnswers)
evaluation_parser = PydanticOutputParser(pydantic_object=SkillEvaluations)
scorecard_parser = PydanticOutputParser(pydantic_object=CandidateScorecard)

print("All Pydantic schemas created.")

All Pydantic schemas created.


In [3]:
transcript_text = """
Interviewer: Can you describe a Python project you completed?
Candidate: I built a script that reads CSV files, cleans missing values,
and calculates summary statistics. I used pandas and tested the output
on sample files.

Interviewer: How do you investigate a software bug?
Candidate: I reproduce the issue, inspect the error message, and test
one possible cause at a time. I document the fix.

Interviewer: Have you worked with SQL databases?
Candidate: I have written basic SELECT queries and used WHERE and ORDER BY.
I have not yet designed a database schema.
"""

skills_to_assess = [
    "Python programming",
    "Debugging",
    "SQL",
    "Database design"
]

print("Interview transcript loaded.")
print("Skills to assess:", skills_to_assess)

Interview transcript loaded.
Skills to assess: ['Python programming', 'Debugging', 'SQL', 'Database design']


In [4]:
answers_prompt = PromptTemplate(
    template="""
Extract each interview question and the candidate's answer from the transcript.

Rules:
- Preserve the meaning of the candidate's answers.
- Do not invent answers or skills.
- Include each question-answer pair in the list.
- Return the required structured format.

Interview transcript:
{transcript_text}

{format_instructions}
""",
    input_variables=["transcript_text"],
    partial_variables={
        "format_instructions": answers_parser.get_format_instructions()
    }
)

answers_chain = answers_prompt | llm | answers_parser

extracted_answers = answers_chain.invoke({
    "transcript_text": transcript_text
})

print("STEP 1 — EXTRACTED ANSWERS")
print(extracted_answers.model_dump_json(indent=2))

STEP 1 — EXTRACTED ANSWERS
{
  "answers": [
    {
      "question": "Can you describe a Python project you completed?",
      "answer": "I built a script that reads CSV files, cleans missing values, and calculates summary statistics. I used pandas and tested the output on sample files."
    },
    {
      "question": "How do you investigate a software bug?",
      "answer": "I reproduce the issue, inspect the error message, and test one possible cause at a time. I document the fix."
    },
    {
      "question": "Have you worked with SQL databases?",
      "answer": "I have written basic SELECT queries and used WHERE and ORDER BY. I have not yet designed a database schema."
    }
  ]
}


In [5]:
evaluation_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a technical interviewer evaluating skill evidence.

Evaluate every requested skill using only the extracted interview answers.

Rules:
- Include one evaluation for every requested skill.
- demonstrated must be true only when clear evidence is present.
- If evidence is absent or insufficient, set demonstrated to false.
- Explain the evidence or gap briefly.
- Do not infer a skill solely from job title or assumptions.
- Do not evaluate protected or personal characteristics.
- Return the required structured format."""
    ),
    (
        "human",
        """Extracted interview answers:
{extracted_answers}

Skills to assess:
{skills_to_assess}

{format_instructions}"""
    )
])

evaluation_chain = evaluation_prompt | llm | evaluation_parser

skill_evaluations = evaluation_chain.invoke({
    "extracted_answers": extracted_answers.model_dump_json(indent=2),
    "skills_to_assess": ", ".join(skills_to_assess),
    "format_instructions": evaluation_parser.get_format_instructions()
})

print("STEP 2 — SKILL EVALUATION")
print(skill_evaluations.model_dump_json(indent=2))

STEP 2 — SKILL EVALUATION
{
  "evaluations": [
    {
      "skill": "Python programming",
      "demonstrated": true,
      "evidence_or_gap": "Built a script that reads CSV files, cleans missing values, and calculates summary statistics using pandas."
    },
    {
      "skill": "Debugging",
      "demonstrated": true,
      "evidence_or_gap": "Described reproducing the issue, inspecting the error message, testing one possible cause at a time, and documenting the fix."
    },
    {
      "skill": "SQL",
      "demonstrated": true,
      "evidence_or_gap": "Wrote basic SELECT queries with WHERE and ORDER BY clauses."
    },
    {
      "skill": "Database design",
      "demonstrated": false,
      "evidence_or_gap": "No evidence of designing a database schema; the candidate explicitly states they have not yet designed one."
    }
  ]
}


In [6]:
scorecard_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a hiring panel coordinator.

Create a structured scorecard from the supplied skill evaluations.

Rules:
- Include every evaluated skill.
- Preserve each demonstrated value and its evidence or gap.
- Base the overall recommendation only on the available interview evidence.
- Do not invent qualifications or make claims beyond the transcript.
- If evidence is insufficient, state that more assessment is needed.
- This scorecard supports human review and must not be treated as an
  automatic hiring decision.
- Return the required structured format."""
    ),
    (
        "human",
        """Skill evaluations:
{skill_evaluations}

{format_instructions}"""
    )
])

scorecard_chain = scorecard_prompt | llm | scorecard_parser

candidate_scorecard = scorecard_chain.invoke({
    "skill_evaluations": skill_evaluations.model_dump_json(indent=2),
    "format_instructions": scorecard_parser.get_format_instructions()
})

print("STEP 3 — CANDIDATE SCORECARD")
print(candidate_scorecard.model_dump_json(indent=2))

STEP 3 — CANDIDATE SCORECARD
{
  "scorecard_rows": [
    {
      "skill": "Python programming",
      "demonstrated": true,
      "evidence_or_gap": "Built a script that reads CSV files, cleans missing values, and calculates summary statistics using pandas."
    },
    {
      "skill": "Debugging",
      "demonstrated": true,
      "evidence_or_gap": "Described reproducing the issue, inspecting the error message, testing one possible cause at a time, and documenting the fix."
    },
    {
      "skill": "SQL",
      "demonstrated": true,
      "evidence_or_gap": "Wrote basic SELECT queries with WHERE and ORDER BY clauses."
    },
    {
      "skill": "Database design",
      "demonstrated": false,
      "evidence_or_gap": "No evidence of designing a database schema; the candidate explicitly states they have not yet designed one."
    }
  ],
  "overall_recommendation": "The candidate demonstrates solid Python programming, debugging, and basic SQL abilities. However, they lack any demons

In [7]:
assert isinstance(extracted_answers, ExtractedAnswers)
assert len(extracted_answers.answers) > 0

assert isinstance(skill_evaluations, SkillEvaluations)
assert len(skill_evaluations.evaluations) == len(skills_to_assess)

evaluated_skills = {item.skill for item in skill_evaluations.evaluations}

for skill in skills_to_assess:
    assert skill in evaluated_skills, f"Missing skill evaluation: {skill}"

assert isinstance(candidate_scorecard, CandidateScorecard)
assert len(candidate_scorecard.scorecard_rows) == len(skills_to_assess)
assert candidate_scorecard.overall_recommendation.strip()

print("Validation passed.")
print("Step 1 extracted interview answers.")
print("Step 2 evaluated all requested skills.")
print("Step 3 generated the structured candidate scorecard.")

Validation passed.
Step 1 extracted interview answers.
Step 2 evaluated all requested skills.
Step 3 generated the structured candidate scorecard.


In [8]:
print("""
Conclusion:
The Interview Transcript to Candidate Scorecard Pipeline contains
three linked steps. Step 1 extracts question-answer pairs using
PromptTemplate and PydanticOutputParser. Step 2 evaluates each
requested skill using ChatPromptTemplate and PydanticOutputParser.
Step 3 creates a structured scorecard with an overall recommendation
using ChatPromptTemplate and PydanticOutputParser.

The scorecard summarizes evidence from the transcript and is intended
to support human review rather than make an automatic hiring decision.
""")


Conclusion:
The Interview Transcript to Candidate Scorecard Pipeline contains
three linked steps. Step 1 extracts question-answer pairs using
PromptTemplate and PydanticOutputParser. Step 2 evaluates each
requested skill using ChatPromptTemplate and PydanticOutputParser.
Step 3 creates a structured scorecard with an overall recommendation
using ChatPromptTemplate and PydanticOutputParser.

The scorecard summarizes evidence from the transcript and is intended
to support human review rather than make an automatic hiring decision.

